# Sesi 4 – Statistika Dasar & Analisis Data

| Keterangan | Detail |
|---|---|
| **Nama Lengkap** | [Farraz Dirgham H] |
| **NIM** | [240401020170] |
| **Kelas** | [IF405] |
| **Mata Kuliah** | Data Science |
| **Topik** | Statistika Dasar & Analisis Data |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)
print('Library siap!')

## 1. Dataset: Nilai Ujian Mahasiswa

In [ ]:
n = 200
df = pd.DataFrame({
    'nama'        : [f'Mahasiswa_{i:03d}' for i in range(1, n+1)],
    'jurusan'     : np.random.choice(['Informatika','Statistika','Matematika','Fisika'], n,
                                     p=[0.35, 0.30, 0.20, 0.15]),
    'nilai_uts'   : np.random.normal(72, 12, n).clip(0, 100).round(1),
    'nilai_uas'   : np.random.normal(75, 10, n).clip(0, 100).round(1),
    'kehadiran'   : np.random.randint(60, 101, n),
    'jam_belajar' : np.random.normal(5, 1.5, n).clip(1, 12).round(1)
})
df['nilai_akhir'] = (df['nilai_uts'] * 0.4 + df['nilai_uas'] * 0.6).round(1)

print(f'Shape: {df.shape}')
df.head()

## 2. Statistika Deskriptif

In [ ]:
print('=== STATISTIKA DESKRIPTIF ===')
print(df.describe())

## 3. Ukuran Pemusatan Data

In [ ]:
data   = df['nilai_akhir']
mean   = data.mean()
median = data.median()
modus  = data.mode()[0]

print(f'Mean   : {mean:.4f}')
print(f'Median : {median:.4f}')
print(f'Modus  : {modus:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(mean,   color='red',    linestyle='--', linewidth=2, label=f'Mean   = {mean:.2f}')
ax.axvline(median, color='green',  linestyle='-.', linewidth=2, label=f'Median = {median:.2f}')
ax.axvline(modus,  color='orange', linestyle=':' , linewidth=2, label=f'Modus  = {modus:.2f}')
ax.set_title('Distribusi Nilai Akhir', fontweight='bold')
ax.set_xlabel('Nilai Akhir')
ax.set_ylabel('Frekuensi')
ax.legend(fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('pemusatan_data.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ukuran Penyebaran Data

In [ ]:
varians = data.var()
std_dev = data.std()
rentang = data.max() - data.min()
q1      = data.quantile(0.25)
q3      = data.quantile(0.75)
iqr     = q3 - q1
cv      = (std_dev / mean) * 100

print(f'Varians (s2)          : {varians:.4f}')
print(f'Standar Deviasi (s)   : {std_dev:.4f}')
print(f'Rentang               : {rentang:.4f}')
print(f'Q1                    : {q1:.4f}')
print(f'Q3                    : {q3:.4f}')
print(f'IQR                   : {iqr:.4f}')
print(f'Koefisien Variasi (%) : {cv:.2f}%')

## 5. Skewness & Kurtosis

In [ ]:
skewness = data.skew()
kurtosis = data.kurtosis()

print(f'Skewness (Kemiringan) : {skewness:.4f}')
print(f'Kurtosis              : {kurtosis:.4f}')

if abs(skewness) < 0.5:
    print('Distribusi: Simetris (mendekati normal)')
elif skewness > 0:
    print('Distribusi: Right-skewed — ekor panjang ke kanan')
else:
    print('Distribusi: Left-skewed — ekor panjang ke kiri')

## 6. Distribusi Frekuensi & Percentile

In [ ]:
bins   = [0, 55, 65, 75, 85, 100]
labels = ['E (<55)', 'D (55-64)', 'C (65-74)', 'B (75-84)', 'A (85-100)']
df['grade'] = pd.cut(df['nilai_akhir'], bins=bins, labels=labels, right=False)

frekuensi = df['grade'].value_counts().sort_index()
rel_freq  = (frekuensi / len(df) * 100).round(2)

tabel = pd.DataFrame({'Frekuensi': frekuensi, 'Persen (%)': rel_freq})
print('=== Tabel Distribusi Frekuensi ===')
print(tabel)

print('\n=== Percentile ===')
for p in [10, 25, 50, 75, 90]:
    print(f'  P{p:2d} = {np.percentile(data, p):.2f}')

## 7. Analisis Korelasi

In [ ]:
numerik = df[['nilai_uts','nilai_uas','kehadiran','jam_belajar','nilai_akhir']]
corr_matrix = numerik.corr()
print(corr_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.8, ax=ax, cbar_kws={'shrink': 0.75})
ax.set_title('Heatmap Korelasi', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

r, p_val = stats.pearsonr(df['jam_belajar'], df['nilai_akhir'])
print(f'\nKorelasi jam_belajar vs nilai_akhir: r={r:.4f}, p={p_val:.6f}')
print('Signifikan' if p_val < 0.05 else 'Tidak Signifikan')

## 8. Analisis Kelompok (Groupby)

In [ ]:
analisis = df.groupby('jurusan').agg(
    N            = ('nama'        , 'count'),
    Rata_UTS     = ('nilai_uts'   , 'mean'),
    Rata_UAS     = ('nilai_uas'   , 'mean'),
    Rata_Akhir   = ('nilai_akhir' , 'mean'),
    Std_Akhir    = ('nilai_akhir' , 'std'),
    Rata_Hadir   = ('kehadiran'   , 'mean')
).round(2)
print(analisis)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar rata-rata nilai per jurusan
warna = ['#2196F3','#4CAF50','#FF5722','#9C27B0']
axes[0].bar(analisis.index, analisis['Rata_Akhir'],
            color=warna, edgecolor='white', alpha=0.85)
for i, (v, e) in enumerate(zip(analisis['Rata_Akhir'], analisis['Std_Akhir'])):
    axes[0].errorbar(i, v, yerr=e, fmt='none', color='black', capsize=5)
    axes[0].text(i, v + e + 0.5, f'{v:.1f}', ha='center', fontweight='bold')
axes[0].set_title('Rata-rata Nilai Akhir per Jurusan', fontweight='bold')
axes[0].set_ylabel('Nilai Akhir')
axes[0].set_ylim(60, 90)
axes[0].spines[['top','right']].set_visible(False)

# Scatter jam belajar vs nilai akhir
jurusan_list = df['jurusan'].unique()
cmap = dict(zip(jurusan_list, warna))
for j in jurusan_list:
    sub = df[df['jurusan'] == j]
    axes[1].scatter(sub['jam_belajar'], sub['nilai_akhir'],
                    color=cmap[j], alpha=0.5, s=40, label=j)
z = np.polyfit(df['jam_belajar'], df['nilai_akhir'], 1)
xr = np.linspace(df['jam_belajar'].min(), df['jam_belajar'].max(), 100)
axes[1].plot(xr, np.poly1d(z)(xr), 'r--', linewidth=2, label='Trend')
axes[1].set_title(f'Jam Belajar vs Nilai Akhir (r={r:.3f})', fontweight='bold')
axes[1].set_xlabel('Jam Belajar per Hari')
axes[1].set_ylabel('Nilai Akhir')
axes[1].legend(fontsize=8)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('analisis_kelompok.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Uji Hipotesis (Independent t-test)

In [ ]:
# H0: Tidak ada perbedaan nilai akhir antara Informatika dan Statistika
grup_a = df[df['jurusan'] == 'Informatika']['nilai_akhir']
grup_b = df[df['jurusan'] == 'Statistika']['nilai_akhir']

t_stat, p_value = stats.ttest_ind(grup_a, grup_b)

print(f'Informatika → Mean: {grup_a.mean():.2f}, n: {len(grup_a)}')
print(f'Statistika  → Mean: {grup_b.mean():.2f}, n: {len(grup_b)}')
print(f't-statistik : {t_stat:.4f}')
print(f'p-value     : {p_value:.6f}')
print('Tolak H0 → Beda SIGNIFIKAN' if p_value < 0.05 else 'Gagal tolak H0 → Tidak berbeda signifikan')

## 10. Kesimpulan

| Konsep Statistika | Fungsi Python |
|---|---|
| Ukuran Pemusatan | `.mean()`, `.median()`, `.mode()` |
| Ukuran Penyebaran | `.std()`, `.var()`, `.quantile()`, IQR |
| Bentuk Distribusi | `.skew()`, `.kurtosis()` |
| Distribusi Frekuensi | `pd.cut()`, `.value_counts()` |
| Korelasi | `.corr()`, `stats.pearsonr()` |
| Analisis Kelompok | `.groupby().agg()` |
| Uji Hipotesis | `stats.ttest_ind()` |

> **Insight:** Statistika adalah "bahasa" dari Data Science. Sebelum membangun model, memahami distribusi dan hubungan antar variabel secara statistik adalah langkah wajib agar model yang dibangun tepat sasaran.